## Kinesis Event Producer Simulator

### What This Notebook Does
This notebook acts as a **stand-in for Amazon Kinesis Data Firehose**. It continuously generates realistic fintech transaction events and drops them as newline-delimited JSON (NDJSON) files into a Unity Catalog Volume every **10 seconds**.

### How to Use
1. **Run cells 1–3** to set up the config, event generator, and start the continuous producer loop
2. **Switch to the consumer notebook** ([Auto Loader Fintech Bronze Ingestion Demo](#notebook-2466812635894401)) in a separate browser tab
3. **Run the Auto Loader cells** there — each time you trigger the stream, it picks up only the new files
4. Come back here to **interrupt the loop** when you’re ready for the schema-evolution demo

> **Keep this notebook running in a separate tab** — it simulates a live upstream data source that never stops producing events.

In [0]:
# ---------------------------------------------------------------------------
# Reset — tear down any resources left over from a previous demo run
# ---------------------------------------------------------------------------
# This ensures a clean slate every time you restart the demo.
# Safe to run even if the resources don't exist yet.
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Demo configuration — change these if you want to use an existing catalog
# ---------------------------------------------------------------------------
CATALOG   = "fintech_demo"
SCHEMA    = "bronze_layer"
VOLUME    = "kinesis_landing"          # simulated Kinesis Firehose delivery path
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.txn_events_bronze"

# Checkpoint stored inside the Volume (fine for demo; use a separate path in prod)
CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint"

# Landing zone where JSON files arrive
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/events"

# 1. Delete the model serving endpoint (not removed by catalog cascade)
try:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    w.serving_endpoints.delete(name="fintech-fraud-scoring")
    print(f"\U0001F5D1\ufe0f  Deleted serving endpoint: fintech-fraud-scoring")
except Exception:
    print(f"\u2139\ufe0f  Serving endpoint not found (skipped)")

# 2. Drop the entire catalog (CASCADE removes all schemas, tables, volumes, models)
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")
print(f"\U0001F5D1\ufe0f  Dropped catalog (cascade): {CATALOG}")

print(f"\n\u2705 Clean slate \u2014 ready for a fresh demo run")

In [0]:
# ---------------------------------------------------------------------------
# Create resources
# ---------------------------------------------------------------------------
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

print(f"\u2705 Catalog  : {CATALOG}")
print(f"\u2705 Schema   : {CATALOG}.{SCHEMA}")
print(f"\u2705 Volume   : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
print(f"\u2705 Landing  : {LANDING_PATH}")
print(f"\u2705 Bronze   : {BRONZE_TABLE}")

In [0]:
# Ensure the landing zone exists (producer notebook creates the Volume)
import os
os.makedirs(LANDING_PATH, exist_ok=True)

print(f"\u2705 Producer configured")
print(f"   Landing zone: {LANDING_PATH}")

In [0]:
import json, uuid, random, os, time
from datetime import datetime, timedelta

# ---------------------------------------------------------------------------
# Realistic fintech transaction event generator
# ---------------------------------------------------------------------------
EVENT_TYPES = ["payment", "transfer", "refund", "fraud_check"]
STATUSES    = ["pending", "completed", "failed", "flagged"]
CHANNELS    = ["mobile_app", "web", "api", "pos_terminal"]
MERCHANTS   = [
    "Acme Coffee Co.", "CloudPay Subscription", "Metro Transit",
    "FinStack Pro", "NeoBank ATM", "GreenGrocer #42", "StreamFlix",
]

def _random_account_id() -> str:
    return f"ACCT-{random.randint(100000, 100500)}"

def generate_event(ts: datetime | None = None, extra_fields: dict | None = None) -> dict:
    """Generate a single realistic fintech transaction event."""
    ts = ts or datetime.utcnow() - timedelta(seconds=random.randint(0, 60))
    event_type = random.choice(EVENT_TYPES)
    evt = {
        "event_id":    str(uuid.uuid4()),
        "event_type":  event_type,
        "timestamp":   ts.isoformat() + "Z",
        "amount":      round(random.uniform(0.50, 19999.99), 2),
        "currency":    "USD",
        "sender_id":   _random_account_id(),
        "receiver_id": _random_account_id(),
        "merchant":    random.choice(MERCHANTS) if event_type == "payment" else None,
        "status":      random.choice(STATUSES),
        "channel":     random.choice(CHANNELS),
    }
    if extra_fields:
        evt.update(extra_fields)
    return evt


def write_event_file(landing_path: str, events_per_file: int = 50, extra_fields: dict | None = None) -> str:
    """Write a single NDJSON file to the landing zone. Returns the file name."""
    batch_ts = datetime.utcnow()
    file_name = f"txn_{batch_ts.strftime('%Y%m%dT%H%M%S')}_{uuid.uuid4().hex[:8]}.json"
    events = [
        json.dumps(generate_event(batch_ts - timedelta(seconds=random.randint(0, 10)), extra_fields))
        for _ in range(events_per_file)
    ]
    file_path = os.path.join(landing_path, file_name)
    with open(file_path, "w") as f:
        f.write("\n".join(events))
    return file_name

print("\u2705 Event generator ready")

### Start Continuous Event Production
The cell below runs an **infinite loop** that drops a new JSON file every **10 seconds** (50 events per file). This mimics a Kinesis Data Firehose delivery stream that flushes micro-batches on a regular interval.

> **To stop:** Click the ■ stop button on this cell, or use **Interrupt** from the cell menu. Then proceed to the Schema Evolution section below.

In [0]:
# ---------------------------------------------------------------------------
# Continuous event production — runs until you interrupt the cell
# ---------------------------------------------------------------------------
# Simulates Kinesis Firehose flushing a new file every 10 seconds
# ---------------------------------------------------------------------------
print("\U0001F680 Starting continuous event production...")
print(f"   Dropping 50-event files every 10 seconds to: {LANDING_PATH}")
print(f"   Press the \u25a0 stop button to interrupt\n")

file_count = 0
total_events = 0

try:
    while True:
        fname = write_event_file(LANDING_PATH, events_per_file=50)
        file_count += 1
        total_events += 50
        now = datetime.utcnow().strftime("%H:%M:%S")
        print(f"   [{now}]  \U0001F4E5 File {file_count}: {fname}  (running total: {total_events:,} events)")
        time.sleep(10)
except KeyboardInterrupt:
    print(f"\n\u23f9\ufe0f  Producer stopped after {file_count} files ({total_events:,} events)")

### Schema Evolution Mode
When you’re ready to demo schema evolution:
1. **Interrupt** the cell above (click \u25a0 stop)
2. **Run the cell below** — it produces events with a **new `risk_score` field** that the original schema didn’t include
3. **Switch to the consumer notebook** and re-run the Auto Loader cell — it will detect and merge the new column automatically

In [0]:
# ---------------------------------------------------------------------------
# Schema evolution producer — events now include a "risk_score" field
# ---------------------------------------------------------------------------
# This simulates the payments team adding fraud-risk metadata upstream
# ---------------------------------------------------------------------------
print("\U0001F680 Starting schema-evolution event production...")
print(f"   Events now include 'risk_score' field (0.0 \u2013 1.0)")
print(f"   Dropping files every 10 seconds to: {LANDING_PATH}")
print(f"   Press the \u25a0 stop button to interrupt\n")

file_count = 0
total_events = 0

try:
    while True:
        extra = {"risk_score": round(random.uniform(0.0, 1.0), 4)}
        fname = write_event_file(LANDING_PATH, events_per_file=50, extra_fields=extra)
        file_count += 1
        total_events += 50
        now = datetime.utcnow().strftime("%H:%M:%S")
        print(f"   [{now}]  \U0001F4E5 File {file_count}: {fname}  (risk_score \u2714, total: {total_events:,} events)")
        time.sleep(10)
except KeyboardInterrupt:
    print(f"\n\u23f9\ufe0f  Schema-evolution producer stopped after {file_count} files ({total_events:,} events)")